# 00. 환경 준비와 테스트

설계 문서 7장 테스트를 Colab GPU 런타임에서 돌린다. `pytest tests/`가 통과해야 작업 완료.

데이터(Drive의 `song2nam/data/`)는 아래 "0. 데이터 내려받기" 셀이 공식 Google Drive에서 받는다(설계 문서 4.6).
- `EGDB/audio_DI/*.wav` — EGDB DI 240곡 (Colab에서 Google 로그인 창이 한 번 뜬다)
- `nam/v3_0_0.wav` — NAM 학습 입력 신호 v3.0.0(NAM 문서의 `input.wav`, MD5로 확인)

In [ ]:
# 공통 준비: Drive 연결, 저장소 받기, 패키지 설치
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
DRIVE = '/content/drive/MyDrive/song2nam'   # 데이터·실행 기록·체크포인트를 둘 Drive 폴더
REPO = '/content/song2nam'
BRANCH = 'main'                               # 작업 브랜치로 바꿔서 쓸 수 있음
os.makedirs(f'{DRIVE}/data', exist_ok=True)
os.makedirs(f'{DRIVE}/runs', exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-b', BRANCH, 'https://github.com/gmeag01/song2nam.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
os.chdir(REPO)
# data/, runs/를 Drive에 연결해서 Colab 런타임이 끊겨도 남게 한다
for d in ['data', 'runs']:
    if os.path.islink(d):
        continue
    subprocess.run(['rm', '-rf', d])
    os.symlink(f'{DRIVE}/{d}', d)

In [ ]:
# Colab에는 torch/torchaudio가 이미 있으므로 나머지만 설치
!grep -vE '^(torch|torchaudio)$' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
import torch; print(torch.__version__, torch.cuda.is_available())

## 0. 데이터 내려받기 (한 번만, 다시 돌리면 이어 받음)

In [ ]:
# 커널 안에서 불러야 Colab Google 로그인(Drive API 목록)이 동작한다
from ampcap.get_data import main as get_data
get_data(['--data', 'data'])

## 1. ⑥ 게인/트레블 구분 테스트 (가장 먼저)

In [ ]:
!python -m pytest tests/test_stats.py -v -s

## 2. 전체 테스트

In [ ]:
!python -m pytest tests/ -v -s

## 3. ⑦ 증류 ESR 테스트 (NAM 학습, 몇 분)

In [ ]:
import os
os.environ['AMPCAP_NAM_INPUT'] = 'data/nam/v3_0_0.wav'
os.environ['AMPCAP_NAM_EPOCHS'] = '30'
!python -m pytest tests/test_distill_eval.py -v -s -k esr

## 4. EGDB 분할 만들기 (한 번만, 곡 단위로 겹치지 않게)

In [ ]:
from ampcap.synth_data import load_split
split = load_split('data/EGDB/audio_DI', 'data/egdb_split.json')
{k: len(v) for k, v in split.items()}

## 5. 대응표 확인: 앰프 계열별 대표 설정을 G로 렌더링해서 들어 보기

In [ ]:
import torch, numpy as np
from IPython.display import Audio, display
from ampcap.graybox import GrayboxAmp
from ampcap.session import compute_prior, FAMILIES
from ampcap.synth_data import di_reader_for_split, normalize_di
reader = di_reader_for_split('data/EGDB/audio_DI', 'val', 'data/egdb_split.json')
di = normalize_di(reader.fixed_clips(3, seed=0)).reshape(1, -1)
G = GrayboxAmp()
for fam in FAMILIES:
    p = compute_prior({'amp': {'family': fam}})
    y = G.render(torch.from_numpy(di), torch.tensor(p.center, dtype=torch.float32)[None])[0].numpy()
    print(fam); display(Audio(y / np.abs(y).max() * 0.8, rate=48000))